Imports

In [1]:
import sys
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_score
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn import tree

# Point to project root
project_root = str(Path.cwd().parent.parent)
if project_root not in sys.path:
    sys.path.append(project_root)

from src.utils.paths import PROCESSED_DATA_DIR
from scipy.stats import randint


In [2]:
# 1. Load the preprocessed data
X_train = pd.read_csv(PROCESSED_DATA_DIR / "X_train.csv")
X_test = pd.read_csv(PROCESSED_DATA_DIR / "X_test.csv")
y_train = pd.read_csv(PROCESSED_DATA_DIR / "y_train.csv").squeeze()
y_test = pd.read_csv(PROCESSED_DATA_DIR / "y_test.csv").squeeze()

# 2. Initialize the model
clf = RandomForestClassifier(random_state=42)

# 3. Train the model
clf.fit(X_train, y_train)

# 4. Make predictions on test set
y_pred = clf.predict(X_test)

# 5. Evaluate the results
print("Accuracy:", accuracy_score(y_test, y_pred))


Accuracy: 0.9623965455199712


### Baseline Random Forest Analysis

The model uses an ensemble of decision trees, where multiple trees are trained on different subsets of the data and their predictions are aggregated.

This reduces variance and improves generalization compared to a single decision tree.

Accuracy: 0.963

### Observations

There is a noticeable class imbalance:

Extroverts (class 0): 4115 samples
Introverts (class 1): 1443 samples

The model performs slightly better on the majority class (extroverts), as seen from higher recall (0.98).

Recall for introverts is lower (0.92), indicating that some introverts are misclassified as extroverts.

The difference between macro and weighted averages suggests that performance is influenced by class imbalance.

### Model Behavior (Random Forest-specific)

Random Forest reduces overfitting by averaging multiple decision trees (bagging).

Each tree is trained on a random subset of the data, improving robustness.

The model can capture complex and non-linear relationships between features.

Compared to a single decision tree:

It is more stable
Less sensitive to noise

<inv>However</inv>:

The model can still be biased toward the majority class in imbalanced datasets.

### Takeaway

The Random Forest model achieves strong performance, comparable to KNN and Decision Tree.

It provides more stability than a single decision tree.

There is still a mild bias toward extroverts, reflected in lower recall for introverts.

Overall, the model is robust and well-suited for this dataset.

In [3]:
print("\nClassification report:", classification_report(y_test, y_pred))


Classification report:               precision    recall  f1-score   support

           0       0.97      0.98      0.97      4115
           1       0.93      0.92      0.93      1443

    accuracy                           0.96      5558
   macro avg       0.95      0.95      0.95      5558
weighted avg       0.96      0.96      0.96      5558



## Classification Report Insights

<inv>Class 0 (Extrovert):</inv>

Precision: 0.97
Recall: 0.98
F1-score: 0.98

The model performs extremely well on this class, correctly identifying most extroverts.

<inv>Class 1 (Introvert):</inv>

Precision: 0.93
Recall: 0.92
F1-score: 0.93

Performance is slightly lower, indicating that some introverts are misclassified as extroverts.

#### Overall:

The model maintains high performance across both classes, but shows a mild imbalance effect favoring extroverts.

## Generalization Analysis

The model achieves high accuracy on the test set (0.963), which is very close to expected performance from training.

There are no clear signs of severe overfitting:

The model generalizes well to unseen data.

Random Forest inherently reduces overfitting compared to a single decision tree through averaging.

However, slight overfitting may still exist due to the complexity of the model.

In [4]:
# 1. Initialize the baseline model
rf_model = RandomForestClassifier(random_state=42)

# 2. Define the Inner and Outer CV
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2)

# 3. Define the Random Forest search space
rf_space = {
    'n_estimators': randint(100, 300), 
    'max_features': ['sqrt', 'log2'],  
    'max_depth': randint(5, 20), 
    'min_samples_split': randint(2, 11), 
    'min_samples_leaf': randint(1, 11)
}

# 4. Set up the Inner Loop (The Random Search)
rf_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=rf_space,
    n_iter=20,          
    scoring='accuracy',
    n_jobs=-1,         
    cv=inner_cv,
    random_state=1
)

# 5. Execute the Outer Loop (Nested CV Evaluation)
rf_nested_scores = cross_val_score(
    estimator=rf_search, 
    X=X_train, 
    y=y_train, 
    cv=outer_cv, 
    scoring='accuracy',
    n_jobs=-1
)

# 6. View the Nested CV results
print(f"\nRF Nested CV Accuracy Scores: {rf_nested_scores}")
print(f"RF Expected Generalization Accuracy: {np.mean(rf_nested_scores):.3f} +/- {np.std(rf_nested_scores):.3f}")

# 7. Extract the model
rf_search.fit(X_train, y_train)
best_rf_model = rf_search.best_estimator_

print(f"Final Best Parameters: {rf_search.best_params_}")

# 8. Evaluate
rf_final_accuracy = best_rf_model.score(X_test, y_test)
print(f"Final Test Set Accuracy: {rf_final_accuracy:.3f}")

/usr/local/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.d


RF Nested CV Accuracy Scores: [0.9676578  0.97000522 0.96791862 0.96138795 0.96608401]
RF Expected Generalization Accuracy: 0.967 +/- 0.003
Final Best Parameters: {'max_depth': 14, 'max_features': 'log2', 'min_samples_leaf': 3, 'min_samples_split': 9, 'n_estimators': 235}
Final Test Set Accuracy: 0.967


## Nested Cross-Validation Analysis (Random Forest)

Expected generalization accuracy: 0.966 ± 0.003

### Interpretation

- The mean accuracy (0.966) is slightly higher than the baseline (~0.963):

This indicates a small but consistent improvement from hyperparameter tuning.

- The standard deviation is low (±0.003):

- Indicates that the model performs consistently across different data splits.

- The individual fold scores range between ~0.960 and ~0.970:

This shows stable performance with no significant drops.

### Why This Matters

<inv>Nested cross-validation provides a more reliable estimate of real-world performance.</inv>

It prevents overfitting caused by tuning and evaluating on the same data.

It ensures that the reported performance is unbiased.

### Takeaway

The Random Forest model demonstrates strong and stable generalization performance.

Hyperparameter tuning slightly improves the model.

The consistency between baseline and nested CV results increases confidence in the model.

### Limitations
- Random Forest is less interpretable compared to simpler models.

- The model can still be slightly biased toward the majority class.

- Training can be computationally expensive due to multiple trees.

- Although overfitting is reduced, it is not completely eliminated.


# Final Conclusion

- The Random Forest model achieves strong performance, with accuracy around <inv>0.963–0.967.</inv>

- The model performs slightly better on the majority class, indicating a mild class imbalance effect.

- It provides stable and robust predictions due to ensemble learning.

- Hyperparameter tuning confirms that the model generalizes well and is not overly sensitive to data splits.

<inv>Overall</inv>, Random Forest is a powerful and reliable model for this dataset.

In [6]:
# Get the tuned Random Forest model
best_rf = rf_search.best_estimator_

# Predictions
y_pred = best_rf.predict(X_test)
y_proba = best_rf.predict_proba(X_test)[:, 1]

# (Optional) Save model for later use
import joblib
joblib.dump(best_rf, "../../models/random_forest.pkl")

['../../models/random_forest.pkl']